[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/055.%20The%20Newsvendor%20Problem/P55-Tier-2_executed.ipynb)

---


# 55. The Newsvendor Problem

## Tier 2: Priority-Based Marginal Analysis Heuristic

### Purpose and scope

This notebook uses a unit-by-unit marginal analysis to choose how many ski jackets Marina should order before demand is known. Demand has a small, specified set of possible values, each with a known probability. The method evaluates the expected profit contribution of each additional unit and stops when that contribution is no longer positive.

The model treats order quantities as nonnegative whole numbers and assumes that every unit sold earns the selling price, while every unsold unit earns the salvage value. It does not include quantity discounts, capacity limits, fixed ordering costs, or uncertainty about the demand probabilities. These assumptions make it possible to isolate the marginal decision taught in this tier.

### Problem data and assumptions

- Selling price: $320 per jacket.
- Purchase cost: $180 per jacket.
- Salvage value: $90 per unsold jacket.
- Demand is 800, 1,000, or 1,200 jackets, with probabilities 0.2, 0.5, and 0.3, respectively.
- The scenario probabilities are nonnegative and sum to 1.

The method compares the expected benefit of ordering one more unit with its expected cost. If the unit is sold, its contribution relative to not ordering it is the selling price minus the purchase cost. If it remains unsold, its contribution is the salvage value minus the purchase cost.

### Mathematical model

For an order quantity $Q$ and realized demand $D$, profit is

$$
\Pi(Q,D) = p\min(D,Q) + v\max(Q-D,0) - cQ.
$$

Here, $p$ is the selling price, $v$ is the salvage value, and $c$ is the purchase cost. The first term is revenue from units sold, the second is the recovery from unsold units, and the last is the cost of all units ordered. With scenarios $d_s$ and probabilities $\pi_s$, expected profit is

$$
\mathbb{E}[\Pi(Q)] = \sum_{s=1}^{S} \pi_s\left[p\min(d_s,Q) + v\max(Q-d_s,0) - cQ\right].
$$

The marginal expected profit of the $q$th unit is the change in expected profit when the order increases from $q-1$ to $q$:

$$
\Delta(q) = \mathbb{E}[\Pi(q)] - \mathbb{E}[\Pi(q-1)]
          = \sum_{s=1}^{S} \pi_s
          \begin{cases}
            p-c, & d_s \ge q, \\
            v-c, & d_s < q.
          \end{cases}
$$

The piecewise expression is evaluated separately for each scenario before its probability-weighted contributions are added. For this problem, a unit that sells contributes $320-$180 = $140, and a unit that is salvaged contributes $90-$180 = -$90. The code function `marginal_profit` computes $\Delta(q)$.

The base case is an order of zero units, with expected profit zero. Increasing the order one unit at a time gives the recurrence

$$
\mathbb{E}[\Pi(Q)] = \mathbb{E}[\Pi(Q-1)] + \Delta(Q),
\qquad \mathbb{E}[\Pi(0)] = 0.
$$

Because the selling price is at least the salvage value, the probability that the $q$th unit sells cannot increase as $q$ increases. Consequently, $\Delta(q)$ is nonincreasing. Adding units while $\Delta(q)>0$ therefore finds an order quantity with maximum expected profit. If a marginal value is exactly zero, more than one quantity can be optimal; stopping at that point selects the smallest maximizing quantity. The implementation evaluates at most $O(SQ)$ scenario contributions for $Q$ candidate units and $S$ demand scenarios.

### Solution sequence

1. Specify the demand scenarios and their probabilities.
2. Calculate the expected marginal profit for each successive unit.
3. Add a unit while its expected marginal profit is positive; stop at the first nonpositive value.
4. Display the marginal-profit pattern and interpret the stopping point.
5. Check the result against the analytical critical-fractile condition.

In [ ]:
import matplotlib.pyplot as plt

print("Library imported")

In [ ]:
# Problem setup: all monetary values are dollars per jacket.
selling_price = 320.0
unit_cost = 180.0
salvage_value = 90.0

# Each pair contains a demand value and its probability.
demand_scenarios = [(800, 0.2), (1000, 0.5), (1200, 0.3)]

assert all(probability >= 0 for _, probability in demand_scenarios)
assert abs(sum(probability for _, probability in demand_scenarios) - 1.0) < 1e-12
assert selling_price >= salvage_value

print("Problem setup complete")

In [ ]:
# Marginal analysis calculation
def marginal_profit(order_qty, scenarios, sp, uc, sv):
    """Return the expected profit contribution of the order_qty-th unit."""
    mp = 0.0
    for demand, prob in scenarios:
        if demand >= order_qty:
            unit_profit = sp - uc
        else:
            unit_profit = sv - uc
        mp += unit_profit * prob
    return mp


# Search successive units through one unit beyond the largest demand scenario.
# This includes a unit that cannot sell in any scenario and has negative contribution.
max_order_qty = max(demand for demand, _ in demand_scenarios) + 1
optimal_qty = 0

for qty in range(1, max_order_qty + 1):
    if marginal_profit(
        qty,
        demand_scenarios,
        selling_price,
        unit_cost,
        salvage_value,
    ) > 0:
        optimal_qty = qty
    else:
        break

print(f"Optimal order quantity: {optimal_qty} jackets")

In [ ]:
# Plot the marginal contribution for every unit considered by the search.
fig, ax = plt.subplots(figsize=(10, 6))

units = range(1, max_order_qty + 1)
marginal_profits = [
    marginal_profit(
        qty,
        demand_scenarios,
        selling_price,
        unit_cost,
        salvage_value,
    )
    for qty in units
]

ax.bar(units, marginal_profits, alpha=0.7, label="Expected marginal profit")
ax.axhline(y=0, color="red", linestyle="--", label="Break-even")
ax.axvline(
    x=optimal_qty,
    color="green",
    linestyle="--",
    label=f"Optimal order quantity: {optimal_qty}",
)
ax.set_xlabel("Unit position in the order, q (jackets)")
ax.set_ylabel("Expected marginal profit per unit ($)")
ax.set_title("Expected Marginal Profit by Unit Position")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

plt.show()
print("Visualization complete")

## Interpretation and analytical check

For unit positions $1$ through $800$, demand exceeds or equals the unit position in every scenario, so the marginal expected profit is $140. For positions $801$ through $1{,}000$, the unit sells in the 1,000- and 1,200-jacket scenarios. Its expected marginal profit is

$$
0.8(140) + 0.2(-90) = 94.
$$

For positions $1{,}001$ through $1{,}200$, the unit sells only in the 1,200-jacket scenario, giving

$$
0.3(140) + 0.7(-90) = -21.
$$

Thus the first 1,000 units have positive expected marginal profit, while the next unit has negative expected marginal profit. The marginal stopping rule selects an order quantity of 1,000 jackets.

As a check, the analytical critical fractile for this newsvendor model is

$$
\frac{p-c}{p-v} = \frac{140}{230} \approx 0.6087.
$$

The cumulative demand probability is 0.2 at 800 jackets and 0.7 at 1,000 jackets. The smallest demand quantity whose cumulative probability reaches the critical fractile is therefore 1,000 jackets, matching the marginal analysis.

The plotted bars represent expected profit contributions of individual units, not total expected profit. The optimal order quantity is the last unit position with a positive contribution; the vertical marker identifies that quantity.